# [WIP] fit_sb2_probmod

In [2]:
import os, numpy as np, pandas as pd
import sys
from glob import glob
sys.path.append('/nexus/posix0/MIA-astro-env/hxr/fvwallauer/MINATO')
import minato
import matplotlib.pyplot as plt
from astropy.constants import c
import os
import minato.binary_population as binSim
from scipy.signal import fftconvolve
from scipy.interpolate import interp1d
import math
from minato import ravel
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'
import numpyro as npro
from jax import numpy as jnp
from jax import random
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive
from minato.ravel import (
    read_spectra, setup_line_dictionary, fit_sb2_probmod, trace_mean,
    pseudo_voigt, rv_shift_wavelength, gaussian, pseudo_voigt, lorentzian, plot_lines_fit
)

JAX 64-bit enabled: True


/home/fvwallauer/.cache/pypoetry/virtualenvs/summerproject-00U4HxGn-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def fit_sb2_probmod(lines, wavelengths, fluxes, f_errors, lines_dic, Hlines, neblines, path, K=2, shift_kms=0,
                    wavelength_type='air', rm_epochs=None):
    """
    Fit SB2 (double-lined spectroscopic binary) spectral lines using a probabilistic
    model with Numpyro. The function interpolates spectral data onto a common grid,
    constructs a Bayesian model for the line profiles, and samples the posterior via
    MCMC (using NUTS).
    
    Parameters:
    -----------
    lines : list
        List of spectral line identifiers (keys from lines_dic) to be fitted.
    wavelengths : list
        List (per epoch) of wavelength arrays.
    fluxes : list
        List (per epoch) of flux arrays.
    f_errors : list
        List (per epoch) of flux error arrays.
    lines_dic : dict
        Dictionary containing spectral line regions, initial centre guesses, etc.
    Hlines : list
        List of lines (subset of `lines`) that are Hydrogen lines.
    neblines : list
        (Currently unused) List of nebular lines.
    path : str
        Path for storing output plots.
    K : int, optional
        Number of components (default 2).
    shift_kms : float, optional
        The overall velocity shift in km/s. For example, use 172 km/s for the SMC.
    wavelength_type : str, optional
        Type of wavelength to use ('air' or 'vacuum'). Default is 'air'.

    Returns:
    --------
    trace : dict
        The MCMC trace (posterior samples).
    x_waves : array (JAX)
        The interpolated wavelength grid for each line and epoch.
    y_fluxes : array (JAX)
        The interpolated fluxes.
    """
    n_lines = len(lines)
    n_epochs = len(wavelengths)
    print('Number of lines:', n_lines)
    print('Number of epochs:', n_epochs)

    # Determine the key to use based on the chosen wavelength type
    key = 'centre' if wavelength_type == 'vacuum' else 'air'

    # Boolean mask for Hydrogen lines (will use Lorentzian instead of Gaussian)
    is_hline = jnp.array([line in Hlines for line in lines])

    # Interpolate fluxes and errors to a common grid
    x_waves_interp = []
    y_fluxes_interp = []
    y_errors_interp = []
    common_grid_length = 200  # Choose a consistent number of points for interpolation

    for line in lines:
        region_start, region_end = lines_dic[line]['region']
        # Shift the region boundaries by shift_kms
        region_start = rv_shift_wavelength(region_start, shift_kms)
        region_end = rv_shift_wavelength(region_end, shift_kms)

        x_waves_line = []
        y_fluxes_line = []
        y_errors_line = []

        for wave_set, flux_set, error_set in zip(wavelengths, fluxes, f_errors):
            mask = (wave_set > region_start) & (wave_set < region_end)
            wave_masked = wave_set[mask]
            flux_masked = flux_set[mask]
            if error_set is not None:
                error_masked = error_set[mask]
            else:
                f_err = compute_flux_err(wave_set, flux_set)
                error_masked = f_err[mask]

            # Interpolate onto a common wavelength grid for this line and epoch
            common_wavelength_grid = np.linspace(wave_masked.min(), wave_masked.max(), common_grid_length)
            interp_flux = interp1d(wave_masked, flux_masked, bounds_error=False, fill_value="extrapolate")(common_wavelength_grid)
            interp_error = interp1d(wave_masked, error_masked, bounds_error=False, fill_value="extrapolate")(common_wavelength_grid)
            x_waves_line.append(common_wavelength_grid)
            y_fluxes_line.append(interp_flux)
            y_errors_line.append(interp_error)

        x_waves_interp.append(x_waves_line)
        y_fluxes_interp.append(y_fluxes_line)
        y_errors_interp.append(y_errors_line)

    # Convert the interpolated lists to JAX arrays (all dimensions now match)
    x_waves = jnp.array(x_waves_interp)       # Shape: (n_lines, n_epochs, common_grid_length)
    y_fluxes = jnp.array(y_fluxes_interp)       # Shape: (n_lines, n_epochs, common_grid_length)
    y_errors = jnp.array(y_errors_interp)       # Shape: (n_lines, n_epochs, common_grid_length)

    # Remove bad epochs along the second axis (axis=1)
    if rm_epochs is not None:
        x_waves = jnp.delete(x_waves, jnp.array(rm_epochs), axis=1)
        y_fluxes = jnp.delete(y_fluxes, jnp.array(rm_epochs), axis=1)
        y_errors = jnp.delete(y_errors, jnp.array(rm_epochs), axis=1)

    # Initial guess for the rest (central) wavelength from lines_dic
    cen_ini = jnp.array([lines_dic[line][key][0] for line in lines])

    epoch_ref = 1    # or choose automatically

    # Define the probabilistic SB2 model
    def sb2_model(λ, fλ, σ_fλ, K, is_hline, Δv_means):
        """
        Numpyro model for SB2 line-profile fitting.

        Parameters:
        -----------
        λ : JAX array
            Interpolated wavelengths with shape (n_lines, n_epochs, ndata).
        fλ : JAX array
            Observed fluxes with shape (n_lines, n_epochs, ndata).
        σ_fλ : JAX array
            Flux uncertainties with shape (n_lines, n_epochs, ndata).
        K : int
            Number of velocity components.
        is_hline : JAX array
            Boolean mask for Hydrogen lines.
        Δv_means : JAX array
            Mean velocity shifts for the K components, with shape (K, 1, 1).

        Returns:
        --------
        Samples are observed via npro.sample("fλ", ...).
        """
        c_kms = c.to('km/s').value  
        nlines, nepochs, ndata = λ.shape

        # Sample continuum level with uncertainty
        logσ_ε = npro.sample('logσ_ε', dist.Uniform(-5, 0))
        σ_ε = jnp.exp(logσ_ε)
        ε = npro.sample('ε', dist.TruncatedNormal(loc=1.0, scale=σ_ε, low=0.7, high=1.1))

        # Define rest wavelengths as a parameter (one per line)
        λ_rest = npro.param("λ_rest", cen_ini)  # Shape: (n_lines,)
        
        # Sample velocity shifts for each epoch and component
        σ_Δv = 200.
        k = 3
        
        gamma = npro.sample("gamma", dist.Normal(loc=shift_kms, scale=σ_Δv))
        theta = npro.sample("theta", dist.Normal(0,1))
        sign = np.tanh(k * theta)

        print(gamma)

        with npro.plate(f'epochs', nepochs, dim=-1):
            s_i = npro.sample("s_i", dist.LogNormal(loc=100, scale=200))
            ds_i = npro.sample("ds_i", dist.LogNormal(loc=100, scale=200))
            dv1_i = sign * s_i
            dv2_i = -sign * (s_i + ds_i)
            # Δv_raw = npro.sample("Δv_raw", dist.Normal(loc=Δv_means, scale=σ_Δv).expand([K, nepochs]))
            # Δv_raw = npro.sample("Δv_raw", dist.Normal(loc=Δv_means, scale=σ_Δv))   # shape (K, nepochs)
            # Δv_sorted = Δv_raw.at[:, epoch_ref].set(jnp.sort(Δv_raw[:, epoch_ref]))
            # Δv_τk = npro.deterministic("Δv_τk", Δv_sorted) 
            Δv_τk = npro.sample("Δv_τk", dist.Normal(loc=Δv_means, scale=σ_Δv))
            dvs = jnp.stack([dv1_i, dv2_i], axis=-2)
            dvs = dvs[None, :, None]



        # Sample amplitudes and widths for each line
        amp1_min, amp1_max = 0.05, 0.3
        amp2_min = 0.01
        with npro.plate(f'lines', nlines, dim=-2):

            # Primary amplitude
            amp0 = npro.sample("amp0", dist.TruncatedNormal(loc=0.18, scale=0.06, low=0.02, high=0.40))
            # Depth ratio
            amp_ratio = npro.sample("amp_ratio", dist.TruncatedNormal(loc=0.60, scale=0.15, low=0.25, high=0.95))
            amp1 = amp_ratio * amp0

            # Stack amplitudes for two components and add extra dimensions for broadcasting
            amp = jnp.stack([amp0, amp1], axis=-3)  # Shape: (2, n_lines)
            amp = amp[:, :, None]  # Shape: (2, n_lines, 1)
            
            # Sample widths for the first component and derive the second component's width.
            # Note: By enforcing wid2 = wid1 + delta_wid (with delta_wid > 0), we ensure that
            # the second component's width is always larger than the first. This constraint is
            # implemented to improve the stability of the fit and prevent label-switching issues,
            # common in multi-component SB2 spectra where lines are often misidentified.
            # For future updates: consider sampling both widths independently and applying
            # a permutation-invariant or post-hoc relabeling scheme if physical evidence suggests
            # that wid2 < wid1 is a possibility.
            wid1 = npro.sample('wid1', dist.Uniform(0.5, 5.0))
            delta_wid = npro.sample('delta_wid', dist.Uniform(0.1, 2.0))
            wid2 = wid1 + delta_wid
            wid = jnp.stack([wid1, wid2], axis=-3)  # Shape: (2, n_lines)
            wid = wid[:, :, None]  # Shape: (2, n_lines, 1)
            
            # Sample widths to use with new Voigt model
            # Sample Gaussian FWHM for component 1
            wid_G1 = npro.sample('wid_G1', dist.Uniform(0.5, 5.0)) # Adjust prior as needed
            # Sample Lorentzian FWHM for component 1
            wid_L1 = npro.sample('wid_L1', dist.Uniform(0.1, 3.0)) # Adjust prior as needed

            # Constrain widths for component 2 (example: wid2 > wid1)
            delta_wid_G = npro.sample('delta_wid_G', dist.Uniform(0.1, 2.0))
            delta_wid_L = npro.sample('delta_wid_L', dist.Uniform(0.05, 1.0))
            wid_G2 = wid_G1 + delta_wid_G
            wid_L2 = wid_L1 + delta_wid_L

            # Stack widths for two components and add extra dimensions for broadcasting
            wid_G = jnp.stack([wid_G1, wid_G2], axis=-3)  # Shape: (2, n_lines)
            wid_L = jnp.stack([wid_L1, wid_L2], axis=-3)  # Shape: (2, n_lines)
            wid_G = wid_G[:, :, None]  # Shape: (2, n_lines, 1)
            wid_L = wid_L[:, :, None]  # Shape: (2, n_lines, 1)

        # Make λ_rest a deterministic variable and reshape for broadcasting
        λ0 = npro.deterministic("λ0", λ_rest)[None, :, None]  # Shape: (1, n_lines, 1)

        # Compute shifted wavelengths for each component and epoch
        print('gamma:', gamma.shape)
        print('dvs:', dvs.shape)
        print('theta:', theta.shape)
        print('sign:', sign.shape)
        print('Δv_τk', Δv_τk.shape)


        μ_2 = λ0 * (1 + Δv_τk / c_kms)  # Broadcasts: (K, n_lines, nepochs) then add an extra axis
        μ = λ0 * (1 + (gamma + dvs) / c_kms)  # Broadcasts: (K, n_lines, nepochs) then add an extra axis
        μ = μ[:, :, :, None]  # Final shape: (K, n_lines, nepochs, 1)

        print('μ_2:', μ_2.shape)
        print('μ:', μ.shape)

        # Prepare the observed wavelengths for model evaluation
        λ_expanded = λ[None, :, :, :]  # Shape: (1, n_lines, nepochs, ndata)
        is_hline_expanded = is_hline[None, :, None, None]  # Shape: (1, n_lines, 1, 1)

        # Compute the model profiles for each component
        gaussian_profile = gaussian(λ_expanded, amp, μ, wid)
        lorentzian_profile = lorentzian(λ_expanded, amp, μ, wid)
        voigt_profile = pseudo_voigt(λ_expanded, amp, μ, wid_G, wid_L)
        # Use Lorentzian for Hydrogen lines, Gaussian otherwise:
        # comp_profile = jnp.where(is_hline_expanded, lorentzian_profile, gaussian_profile)
        comp_profile = jnp.where(is_hline_expanded, lorentzian_profile, voigt_profile)
        # comp_profile = gaussian_profile
        # comp_profile = voigt_profile
        Ck = npro.deterministic("C_λk", comp_profile)

        # Sum over components and add continuum to yield the predicted flux
        fλ_pred = npro.deterministic("fλ_pred", ε + Ck.sum(axis=0))
        # Likelihood: compare predicted flux with observed flux
        npro.sample("fλ", dist.Normal(fλ_pred, σ_fλ), obs=fλ)

    # ------------------------
    # MCMC Sampling Procedure
    # ------------------------
    comp_sep = 200.
    Δv_means = jnp.array([shift_kms - comp_sep/2, shift_kms + comp_sep/2]).reshape(K, 1, 1)
    print(f"\nFitting with Δv_means: {Δv_means}")

    # Set a fixed random key (you can change this seed if desired)
    rng_key = random.PRNGKey(0)



    # prior = Predictive(sb2_model, num_samples=200)
    # s = prior(rng_key, λ=x_waves, fλ=y_fluxes, σ_fλ=y_errors,
    #         K=K, is_hline=is_hline, Δv_means=Δv_means)
    # dv = s["Δv_τk"]        # shape (200, K, nepochs)
    # assert (dv[:,0,epoch_ref] < dv[:,1,epoch_ref]).all()




    kernel = NUTS(sb2_model)
    mcmc = MCMC(kernel, num_warmup=1000, num_chains=4, num_samples=2000)
    mcmc.run(rng_key, extra_fields=("potential_energy",), 
             λ=x_waves, fλ=y_fluxes, σ_fλ=y_errors, K=K, is_hline=is_hline, Δv_means=Δv_means)

    # Evaluate the mean log posterior probability (for diagnostic purposes)
    potential_energy = mcmc.get_extra_fields()['potential_energy']
    log_probs = -potential_energy  # Convert potential energy to log probability
    log_prob = np.mean(log_probs)
    print(f"Mean log posterior probability: {losg_prob}")

    # Get the MCMC trace (posterior samples)
    trace = mcmc.get_samples()

    # fλ_pred_plot = False
    # if fλ_pred_plot:
    #     num_prior_samples = 200
    #     prior_predictive = Predictive(sb2_model, num_samples=num_prior_samples)
    #     prior_samples = prior_predictive(rng_key, λ=x_waves, σ_fλ=y_errors, K=K, is_hline=is_hline, Δv_means=Δv_means)
    #     # Plotting code for prior predictive checks can go here...
    #     plt.show()

    # ------------------------
    # Plotting the Fitted Profiles
    # ------------------------
    plot_lines_fit(wavelengths, lines, x_waves, y_fluxes, n_epochs, trace, lines_dic, shift_kms, comp_sep, path)

    return trace, x_waves, y_fluxes


In [4]:

# ------------------ 1. Load one system ------------------
sys_dir = "SB2_sys019_snr30"
spec_files = sorted(glob(os.path.join(sys_dir, "*epoch??.txt")))

wavs, fluxes, ferrs, names, jds = read_spectra(
    spec_files, path=sys_dir + "/", file_type="txt",
    instrument="FLAMES", SB2=True
)

lines = [4026, 4144, 4388, 4471]
lines_dic = setup_line_dictionary()

# ------------------ 2. Run fit ------------------
trace, x_waves, y_fluxes = fit_sb2_probmod(
    lines, wavs, fluxes, ferrs, lines_dic,
    Hlines=[4102, 4340, 4861, 6562], neblines=[],
    path="./", shift_kms=0.0
)



Number of lines: 4
Number of epochs: 12

Fitting with Δv_means: [[[-100.]]

 [[ 100.]]]


/tmp/ipykernel_3480650/1454111782.py:275: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(kernel, num_warmup=1000, num_chains=4, num_samples=2000)


-1.7688590123939765
gamma: ()
dvs: (1, 2, 1, 12)
theta: ()
sign: ()
Δv_τk (2, 1, 12)
μ_2: (2, 4, 12)
μ: (1, 2, 4, 1, 12)


/nexus/posix0/MIA-astro-env/hxr/fvwallauer/MINATO/minato/ravel.py:545: SyntaxWarning: invalid escape sequence '\d'
  'region': [4080, 4122], 'centre': [4102.8991, 0.0024], 'air': [4101.7414, 0.0024], 'wid_ini': 6, 'title': 'H$\delta$'},
/nexus/posix0/MIA-astro-env/hxr/fvwallauer/MINATO/minato/ravel.py:547: SyntaxWarning: invalid escape sequence '\g'
  'region': [4316, 4366], 'centre': [4341.691, 0.003],   'air': [4340.471, 0.003],   'wid_ini': 7, 'title': 'H$\gamma$'},
/nexus/posix0/MIA-astro-env/hxr/fvwallauer/MINATO/minato/ravel.py:554: SyntaxWarning: invalid escape sequence '\l'
  4009: { 'region': [4001, 4014], 'centre': [4010.3899037, 0.0000011], 'air': [4009.256516, 0.000020], 'wid_ini': 3, 'title': 'He I $\lambda$4009'},
/nexus/posix0/MIA-astro-env/hxr/fvwallauer/MINATO/minato/ravel.py:555: SyntaxWarning: invalid escape sequence '\l'
  4026: { 'region': [4013, 4039], 'centre': [4027.36003, 0.00004],     'air': [4026.22221, 0.00004],   'wid_ini': 3, 'title': 'He I $\lambda$4026'}

ValueError: Incompatible shapes for broadcasting: shapes=[(1, 4, 12, 200), (1, 2, 4, 1, 12)]